# Random rollout on CartPole

This notebook shows the smallest end-to-end loop: build an environment from `EnvConfig`, sample random actions, and watch the rollout as an embedded MP4.

## What mouse-gym does

Most Gymnasium environments are **episodic**: an agent acts until termination or truncation, then the caller must call `reset()` before the next episode can begin. That interface works well when each episode is an independent sample. It becomes awkward when the experiment is about behavior *across* multiple episodes — where what an agent learns or observes in one episode should influence the next.

**mouse-gym** converts episodic Gymnasium environments into **continuing environments**. You call `step()` in a loop, forever. When an episode or task ends, the next `step()` returns a **reset frame** — initial observation, `time=0`, `done=0` — with the input ignored. There is no `SingleEnv.reset()`, including at task boundaries; mouse-gym calls the underlying Gymnasium `reset()` internally on that step.

Three design decisions make this more than a thin wrapper:

- **Reset frames are first-class outputs.** The reset observation appears in the same output record shape as every other step, with `time=0`, `done=0`, and the configured `reset_reward`. Training code sees a uniform stream.
- **Episode structure stays visible.** The `done` field uses integer codes so transitions, truncations, and reset frames are all distinguishable: `0` = running, `1` = terminated (Gymnasium `terminated=True`), `2` = truncated (Gymnasium `truncated=True`). Reset frames always emit `done=0`.

## Step outputs

Each `step()` returns one output `dict`:

| Field | Type | Notes |
|-------|------|-------|
| `observation` | array | Native shape from the env (e.g. `(4,)` for CartPole). |
| `reward` | float32 array | Raw env reward. Uses `reset_reward` (default `0`) on reset frames. |
| `done` | int64 array | `0` running · `1` ep. terminated · `2` ep. truncated · `3` task terminated · `4` task truncated. |
| `time` | int64 array | Step index within the current episode (0-based; `0` on reset frames). |
| `episode_index` | int | Episode counter. |
| `task_index` | int | Task counter. |

## Introspecting contracts: `input_spec` and `output_spec`

`env.input_spec` and `env.output_spec` describe the stable construction-time contract. The Gymnasium `info` dict is forwarded verbatim on each step output under `info`.

```python
ispec = env.input_spec
ospec = env.output_spec

ispec.action.dtype   # np.int64 (discrete) or np.float32 (continuous)
ispec.action.shape   # () for scalar; (n,) for multi-dimensional

ospec.observation.dtype   # np.float32 for CartPole
ospec.observation.shape   # (4,) for CartPole
ospec.reward.dtype        # np.float32
```

Use these to allocate replay buffers, build network heads, or validate shapes before running any rollout.

## Imports

In [1]:
import os

import matplotlib.animation as animation
import matplotlib.pyplot as plt
from IPython.display import HTML, display

# Some Gymnasium envs render via pygame; run headless when no display is available.
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")

from mouse_gym import EnvConfig, make_env, make_group_env

## Configure the environment

`EnvConfig` is the configuration object passed to `make_env`. One config creates one `SingleEnv`; pass a list to `make_group_env` for multiple instances.

Required: `reset_seed`.

Provide exactly one of `id` (Gymnasium env id) or `env_fn` (zero-arg factory returning a Gymnasium env).

| Field | Purpose |
|-------|---------|
| `reset_seed` | Seed for mouse-gym's internal Gymnasium reset stream |
| `episodes_per_task` | Number of episodes before a task boundary (`done=3/4`) fires (default `0` = unlimited) |

Everything else is optional. Notable optional fields: `name` sets the display name (defaults to `id`, or the factory's `__name__` for `env_fn` configs); `kwargs` passes arguments to `gym.make` (`id` configs only); `env_fn` builds a custom env instead of `id`.

`make_env` returns a `SingleEnv`. `env.name` is the env instance name — e.g. `"CartPole-v1"` here when `name` is not set.

In [2]:
cfg = EnvConfig(
    id="CartPole-v1",
    reset_seed=0,
    episodes_per_task=100,
    kwargs={"render_mode": "rgb_array"},
)
env = make_env(cfg)

## Rollout

Run CartPole for 200 random-action steps. There is no public `reset()` call — `step()` handles episode boundaries internally. Frames are captured each step and displayed as an embedded MP4 at the end.

Each iteration prints the action dict (`inp`) and the full output dict (`output`) so you can see the `done`, `time`, `reward`, and `episode_index` values live. Watch `time` reset to `0` and `done` go non-zero whenever CartPole terminates.

In [3]:
frames = []
for _ in range(200):
    inp = env.sample_random_input()
    output = env.step(inp)
    frames.append(env.render()[0])
    print(inp)
    print(output)


def display_env_replay(name, frames, *, fps=20):
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.axis("off")
    ax.set_title(name, fontsize=10)
    img = ax.imshow(frames[0])

    def update(t):
        img.set_data(frames[t])
        return (img,)

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(frames),
        interval=1000 / fps,
        blit=True,
    )
    plt.close(fig)
    display(HTML(ani.to_html5_video()))


display_env_replay(env.name, frames)

{'action': array(1)}
{'time': array(0), 'reward': array(0., dtype=float32), 'done': array(0), 'episode_index': 0, 'task_index': 0, 'observation': array([-0.01467975,  0.017991  ,  0.03756642, -0.02614838], dtype=float32), 'info': {}}
{'action': array(1)}
{'time': array(1), 'reward': array(1., dtype=float32), 'done': array(0), 'episode_index': 0, 'task_index': 0, 'observation': array([-0.01431993,  0.21255465,  0.03704344, -0.306746  ], dtype=float32), 'info': {}}
{'action': array(1)}
{'time': array(2), 'reward': array(1., dtype=float32), 'done': array(0), 'episode_index': 0, 'task_index': 0, 'observation': array([-0.01006884,  0.4071297 ,  0.03090853, -0.58752006], dtype=float32), 'info': {}}
{'action': array(0)}
{'time': array(3), 'reward': array(1., dtype=float32), 'done': array(0), 'episode_index': 0, 'task_index': 0, 'observation': array([-0.00192624,  0.21158883,  0.01915812, -0.2852632 ], dtype=float32), 'info': {}}
{'action': array(0)}
{'time': array(4), 'reward': array(1., dtyp

{'action': array(1)}
{'time': array(21), 'reward': array(1., dtype=float32), 'done': array(0), 'episode_index': 2, 'task_index': 0, 'observation': array([ 0.00952022,  0.6032001 , -0.11011847, -1.1274532 ], dtype=float32), 'info': {}}
{'action': array(1)}
{'time': array(22), 'reward': array(1., dtype=float32), 'done': array(0), 'episode_index': 2, 'task_index': 0, 'observation': array([ 0.02158422,  0.7995787 , -0.13266753, -1.4525465 ], dtype=float32), 'info': {}}
{'action': array(0)}
{'time': array(23), 'reward': array(1., dtype=float32), 'done': array(0), 'episode_index': 2, 'task_index': 0, 'observation': array([ 0.0375758 ,  0.60631233, -0.16171846, -1.2040843 ], dtype=float32), 'info': {}}
{'action': array(1)}
{'time': array(24), 'reward': array(1., dtype=float32), 'done': array(0), 'episode_index': 2, 'task_index': 0, 'observation': array([ 0.04970204,  0.80311227, -0.18580015, -1.5427707 ], dtype=float32), 'info': {}}
{'action': array(1)}
{'time': array(25), 'reward': array(1.,

{'action': array(1)}
{'time': array(16), 'reward': array(1., dtype=float32), 'done': array(0), 'episode_index': 6, 'task_index': 0, 'observation': array([ 0.04739637,  0.44580284, -0.0599545 , -0.6822455 ], dtype=float32), 'info': {}}
{'action': array(1)}
{'time': array(17), 'reward': array(1., dtype=float32), 'done': array(0), 'episode_index': 6, 'task_index': 0, 'observation': array([ 0.05631243,  0.6417039 , -0.07359941, -0.99318516], dtype=float32), 'info': {}}
{'action': array(1)}
{'time': array(18), 'reward': array(1., dtype=float32), 'done': array(0), 'episode_index': 6, 'task_index': 0, 'observation': array([ 0.06914651,  0.8377292 , -0.09346312, -1.3080459 ], dtype=float32), 'info': {}}
{'action': array(1)}
{'time': array(19), 'reward': array(1., dtype=float32), 'done': array(0), 'episode_index': 6, 'task_index': 0, 'observation': array([ 0.0859011 ,  1.033903  , -0.11962403, -1.6284604 ], dtype=float32), 'info': {}}
{'action': array(0)}
{'time': array(20), 'reward': array(1.,

## Inspect specs

`input_spec` and `output_spec` describe the full contract before any steps. Use them to allocate buffers, build network heads, or validate shapes.

In [4]:
ispec = env.input_spec
ospec = env.output_spec

print("Input spec:")
print(f"  action  dtype={ispec.action.dtype}  shape={ispec.action.shape}")
print()
print("Output spec:")
print(f"  observation  dtype={ospec.observation.dtype}  shape={ospec.observation.shape}")
print(f"  reward       dtype={ospec.reward.dtype}        shape={ospec.reward.shape}")
print(f"  done         dtype={ospec.done.dtype}          shape={ospec.done.shape}")
print(f"  time         dtype={ospec.time.dtype}          shape={ospec.time.shape}")

Input spec:
  action  dtype=int64  shape=()

Output spec:
  observation  dtype=float32  shape=(4,)
  reward       dtype=float32        shape=()
  done         dtype=int64          shape=()
  time         dtype=int64          shape=()


## Cleanup

In [5]:
env.close()